# Notebook 2 — Alur Proyek FERQUEUE (SSD MobileNet V2 FPNLite 320×320)

Notebook Google Colab ini membedah secara transparan seluruh tahapan **Computer Vision yang benar-benar berjalan di proyek FERQUEUE** menggunakan model hasil trainingmu sendiri (`pcv-model-colab-v3-ckpt-35.zip` / `SavedModel`, 3 kelas: `Mobil`, `Truk`, `Bus`, dilatih pada 904 train / 387 test selama 30 epoch).

### Keunggulan Notebook Ini
* **Tidak perlu instalasi TF Object Detection API / protobuf / venv** — cukup pakai TensorFlow bawaan Google Colab untuk memuat `SavedModel` (`saved_model.pb`) dan membedah isi *Feature Maps* langsung dari bobot modelmu sendiri!
* **Memvisualisasikan tahapan otomatis MobileNetV2**:
  1. **Preprocessing** (Eksternal `processing.py` vs Internal SSD: Resize `320×320`, Normalisasi `[-1, 1]`, & Augmentasi training).
  2. **Ekstraksi Fitur Otomatis** — Mengekstrak **Feature Maps nyata** dari *Low-Level* (`160×160×32`), *Mid-Level* (`80×80×96`), hingga 5 skala **FPNLite (P3–P7)** + **Activation Heatmap Overlay**.
  3. **Deteksi & Post-Processing (`postprocessing.py`)** — Membandingkan **Raw Candidate Boxes (12.804 anchor)** vs **Confidence Filtering (`>= 0.5`)** vs **Hasil Akhir NMS (`IoU <= 0.45`)**.
  4. **Segmentasi Pasca-Deteksi (`processing.py`)** — Menerapkan **GrabCut + Morfologi** pada ROI bounding box hasil prediksi (jujur terhadap arsitektur: SSD memprediksi bbox dari gambar utuh, GrabCut memisahkan *foreground* untuk review/analisis bentuk).
  5. *(Opsional)* **Kalibrasi Koin 50 Yen (`side_measurement.py`)** — Segmentasi Otsu + Contour + Fit Ellipse untuk estimasi panjang & tinggi kendaraan (cm).

## Cell 1 — Setup Library & Folder Output Presentasi

In [ ]:
import json
import math
import shutil
import zipfile
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # type: ignore[name-defined]
except NameError:
    display = print

OUT_DIR = Path("/content/outputs_presentasi/nb2_ferqueue_ssd") if IN_COLAB else Path("./outputs_presentasi/nb2_ferqueue_ssd")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_WORK_DIR = Path("/content/ferqueue_model_viz") if IN_COLAB else Path("./outputs_presentasi/_model_cache")
MODEL_WORK_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"


def save_fig(fig, filename: str):
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Tersimpan: {path}")
    return path


print(f"TensorFlow={tf.__version__} | OpenCV={cv2.__version__} | GPU={len(tf.config.list_physical_devices('GPU')) > 0}")

## Cell 2 — Muat Model FERQUEUE (`pcv-model-colab-v3-ckpt-35.zip`) & Gambar Uji

Pilih sumber model lewat `MODEL_SOURCE`:
* `"upload_zip"` *(paling praktis di Colab)*: unggah file `outputs/pcv-model-colab-v3-ckpt-35.zip` (~10 MB) langsung dari laptop.
* `"drive"`: baca ZIP atau folder `saved_model` dari Google Drive (ubah `DRIVE_MODEL_PATH`).
* `"local"`: fallback jika dijalankan lokal di root repo FERQUEUE.

In [ ]:
MODEL_SOURCE = "upload_zip"   # Pilihan: "upload_zip" | "drive" | "local"
DRIVE_MODEL_PATH = "/content/drive/MyDrive/ferqueue/pcv-coco-colab-v3/pcv-model-colab-v3-ckpt-35.zip"
LOCAL_MODEL_PATH = "models/mobilenet/runs/pcv-receiver-v3-unfrozen/exports/imported-ckpt-35"

IMAGE_SOURCE = "upload"       # Pilihan: "upload" | "drive" | "local"
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/ferqueue/sample_images"
LOCAL_IMAGE_DIR = "outputs/test-samples"
ACTIVE_INDEX = 0              # Indeks gambar utama yang dibedah di Cell 3-7

DEFAULT_CLASSES = ["Mobil", "Truk", "Bus"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def resolve_saved_model(src_mode: str):
    extract_dir = MODEL_WORK_DIR / "extracted"
    meta = {
        "classes": DEFAULT_CLASSES,
        "model": "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8",
        "checkpoint": "ckpt-35",
        "preprocessing": {"denoise_kernel": 0, "clahe_clip_limit": 0, "clahe_grid_size": 8, "gamma": 1.0},
    }

    if src_mode == "upload_zip" and IN_COLAB:
        print("Silakan unggah file ZIP model (misal: pcv-model-colab-v3-ckpt-35.zip)...")
        up = colab_files.upload()
        zip_name, content = next(iter(up.items()))
        zip_file = MODEL_WORK_DIR / Path(zip_name).name
        zip_file.write_bytes(content)
        target_path = zip_file
    elif src_mode == "drive" and IN_COLAB:
        drive.mount("/content/drive")
        target_path = Path(DRIVE_MODEL_PATH)
    else:
        target_path = Path(LOCAL_MODEL_PATH)
        if not target_path.exists():
            alt_zip = Path("outputs/pcv-model-colab-v3-ckpt-35.zip")
            if alt_zip.is_file():
                target_path = alt_zip

    if target_path.is_file() and target_path.suffix.lower() == ".zip":
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target_path) as zf:
            zf.extractall(extract_dir)
        if (extract_dir / "model.json").is_file():
            meta.update(json.loads((extract_dir / "model.json").read_text(encoding="utf-8")))
        saved_dir = extract_dir / "saved_model"
    elif target_path.is_dir():
        saved_dir = target_path / "saved_model" if (target_path / "saved_model/saved_model.pb").is_file() else target_path
        export_json = target_path.parent.parent / "export.json"
        if export_json.is_file():
            meta.update(json.loads(export_json.read_text(encoding="utf-8")))
    else:
        raise FileNotFoundError(f"Model tidak ditemukan di: {target_path}")

    if not (saved_dir / "saved_model.pb").is_file():
        raise FileNotFoundError(f"saved_model.pb tidak ditemukan di dalam: {saved_dir}")
    return saved_dir, meta


saved_model_dir, model_meta = resolve_saved_model(MODEL_SOURCE)
CLASSES = model_meta["classes"]
MODEL_PREPROCESSING = model_meta.get("preprocessing", {})

print(f"Memuat SavedModel dari: {saved_model_dir} ...")
t0 = perf_counter()
loaded_tf_model = tf.saved_model.load(str(saved_model_dir))
detector_fn = loaded_tf_model.signatures["serving_default"]
print(f"Model siap dalam {perf_counter() - t0:.2f} detik!")
print(f"Arsitektur : {model_meta['model']} | Checkpoint: {model_meta['checkpoint']}")
print(f"Kelas      : {CLASSES}")
print(f"Preprocess : {MODEL_PREPROCESSING}")

# Muat gambar uji
UPLOAD_DIR = OUT_DIR / "_inputs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
image_paths = []
if IMAGE_SOURCE == "upload" and IN_COLAB:
    print("\nSilakan unggah 1 atau lebih gambar kendaraan uji (JPG/PNG)...")
    uploaded_imgs = colab_files.upload()
    for fname, content in uploaded_imgs.items():
        dst = UPLOAD_DIR / Path(fname).name
        dst.write_bytes(content)
        if dst.suffix.lower() in IMAGE_EXTS:
            image_paths.append(dst)
elif IMAGE_SOURCE == "drive" and IN_COLAB:
    drive.mount("/content/drive")
    image_paths = sorted(p for p in Path(DRIVE_IMAGE_DIR).iterdir() if p.suffix.lower() in IMAGE_EXTS)
else:
    if Path(LOCAL_IMAGE_DIR).is_dir():
        image_paths = sorted(p for p in Path(LOCAL_IMAGE_DIR).iterdir() if p.suffix.lower() in IMAGE_EXTS)

if not image_paths:
    raise FileNotFoundError("Belum ada gambar uji yang dimuat.")


def load_pil_rgb(path: Path) -> Image.Image:
    with Image.open(path) as img:
        return ImageOps.exif_transpose(img).convert("RGB")


loaded_images = [(p.name, load_pil_rgb(p)) for p in image_paths]
ACTIVE_INDEX = min(ACTIVE_INDEX, len(loaded_images) - 1)
active_name, active_pil = loaded_images[ACTIVE_INDEX]
img_rgb = np.asarray(active_pil, dtype=np.uint8)
print(f"\nGambar aktif [{ACTIVE_INDEX}]: {active_name} ({img_rgb.shape[1]}x{img_rgb.shape[0]} px)")

## Cell 3 — Tahap 1: Preprocessing Proyek FERQUEUE (Eksternal & Internal SSD)

Pada proyek FERQUEUE, preprocessing terbagi menjadi **2 lapisan**:
1. **Preprocessing Eksternal (`ferqueue_cv/processing.py`)**:
   - EXIF orientation fix + konversi RGB (`uint8`, `0–255`).
   - Tersedia filter *geometry-preserving* opsional (Gaussian Blur, CLAHE pada ruang warna LAB, Gamma LUT). Pada model v3 final (`receiver-v3.json`), ketiganya diset **baseline (`0 / 0 / 1.0`)** agar tekstur asli kendaraan tetap utuh.
2. **Preprocessing Internal SSD MobileNet V2 (`pipeline.config`)**:
   - `fixed_shape_resizer`: Resize ke **`320×320` piksel**.
   - **Normalisasi piksel** ke rentang **`[-1.0, +1.0]`** dengan rumus $(x / 127.5) - 1.0$.
   - Saat *training* (`coco-v3-colab.json`), juga diterapkan augmentasi `random_horizontal_flip` dan `random_adjust_brightness (max_delta=0.1)`.

In [ ]:
def preprocess_image_ferqueue(image: Image.Image, settings=None) -> Image.Image:
    """Persis mengikuti apps/computer-vision/src/ferqueue_cv/processing.py."""
    settings = settings or {}
    image = image.convert("RGB")
    kernel = settings.get("denoise_kernel", 0)
    clip = settings.get("clahe_clip_limit", 0)
    gamma = settings.get("gamma", 1.0)
    if not kernel and not clip and gamma == 1.0:
        return image
    pixels = np.asarray(image).copy()
    if kernel:
        pixels = cv2.GaussianBlur(pixels, (kernel, kernel), 0)
    if clip:
        lab = cv2.cvtColor(pixels, cv2.COLOR_RGB2LAB)
        grid = settings.get("clahe_grid_size", 8)
        lab[:, :, 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(lab[:, :, 0])
        pixels = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    if gamma != 1.0:
        table = np.round(255 * (np.arange(256) / 255) ** gamma).astype("uint8")
        pixels = cv2.LUT(pixels, table)
    return Image.fromarray(pixels)


# Terapkan preprocessing model aktif (v3 baseline: RGB asli)
proc_pil = preprocess_image_ferqueue(active_pil, MODEL_PREPROCESSING)
proc_rgb = np.asarray(proc_pil, dtype=np.uint8)

# Simulasi internal SSD MobileNetV2: Resize 320x320 & Normalisasi [-1, 1]
resized_320 = cv2.resize(proc_rgb, (320, 320), interpolation=cv2.INTER_LINEAR)
norm_m1_p1 = (resized_320.astype(np.float32) / 127.5) - 1.0

# Simulasi augmentasi training (horizontal flip + brightness +0.1)
aug_flip = cv2.flip(resized_320, 1)
aug_bright = np.clip(resized_320.astype(np.float32) + 0.1 * 255.0, 0, 255).astype(np.uint8)

fig = plt.figure(figsize=(16, 7.6))
gs = fig.add_gridspec(2, 4, height_ratios=[1.3, 1.0], hspace=0.35, wspace=0.2)

panels = [
    (f"1. Input RGB Asli\n({img_rgb.shape[1]}x{img_rgb.shape[0]}, [0..255])", img_rgb),
    ("2. Internal SSD: Resize\n(320x320, fixed_shape_resizer)", resized_320),
    ("3. Augmentasi Training\n(Random Horizontal Flip)", aug_flip),
    ("4. Augmentasi Training\n(Brightness Delta ±0.1)", aug_bright),
]
for i, (title, im) in enumerate(panels):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(im)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

ax_h1 = fig.add_subplot(gs[1, 0:2])
ax_h1.hist(resized_320.ravel(), bins=64, range=(0, 256), color="#4c72b0", alpha=0.85)
ax_h1.set_title("Distribusi Piksel Sebelum Normalisasi (uint8: 0 s/d 255)")
ax_h1.set_xlabel("Nilai Piksel")
ax_h1.set_ylabel("Frekuensi")
ax_h1.grid(alpha=0.25)

ax_h2 = fig.add_subplot(gs[1, 2:4])
ax_h2.hist(norm_m1_p1.ravel(), bins=64, range=(-1.0, 1.0), color="#55a868", alpha=0.85)
ax_h2.set_title("Distribusi Piksel Masuk MobileNetV2 (float32: -1.0 s/d +1.0)")
ax_h2.set_xlabel("Nilai Ternormalisasi (pixel / 127.5 - 1.0)")
ax_h2.set_ylabel("Frekuensi")
ax_h2.grid(alpha=0.25)

fig.suptitle(f"Tahap 1: Preprocessing Proyek FERQUEUE — {active_name}", fontsize=14, fontweight="bold")
save_fig(fig, "01_preprocessing_ferqueue.png")
plt.show()

print(f"Contoh nilai piksel [0,0] RGB mentah      : {resized_320[0, 0].tolist()}")
print(f"Contoh nilai piksel [0,0] ternormalisasi  : {np.round(norm_m1_p1[0, 0], 4).tolist()}")

## Cell 4 — Tahap 2: Membedah Ekstraksi Fitur Otomatis (MobileNetV2 + FPNLite P3–P7)

Walaupun ekstraksi fitur di CNN berjalan otomatis di dalam model, kita bisa **mengekstrak langsung tensor *Feature Maps* intermediate** dari `SavedModel` hasil trainingmu sendiri menggunakan *graph pruning* TensorFlow:
* **Layer Awal MobileNetV2 (`160×160×32` & `80×80×96`)**: mengekstrak fitur *low-level* seperti tepi, garis bodi, jendela, dan tekstur roda.
* **FPNLite Piramida Fitur (`P3` s/d `P7`, masing-masing 128 channel)**:
  - **P3 (`40×40×128`)** → resolusi tinggi untuk mendeteksi kendaraan kecil / jauh.
  - **P4 (`20×20×128`) & P5 (`10×10×128`)** → kendaraan berukuran sedang.
  - **P6 (`5×5×128`) & P7 (`3×3×128`)** → fitur semantik global untuk kendaraan besar (`Truk` / `Bus`).
* **Activation Heatmap Overlay**: menumpuk rata-rata energi aktivasi FPN di atas gambar asli untuk melihat **area mana pada kendaraan yang paling diperhatikan model**.

In [ ]:
# Bangun sub-graph ekstraktor feature map langsung dari SavedModel FERQUEUE
print("Mengekstrak graph internal MobileNetV2 + FPNLite dari SavedModel (tunggu ~5-15 detik)...")
frozen_fn = convert_variables_to_constants_v2(detector_fn, lower_control_flow=False)
all_tensor_names = [out.name for op in frozen_fn.graph.get_operations() for out in op.outputs]

LAYER_PATTERNS = [
    ("Low-Level (160x160x32)", "expanded_conv_depthwise_relu/Relu6:0"),
    ("Mid-Level (80x80x96)", "block_1_depthwise_relu/Relu6:0"),
    ("FPN P3 (40x40x128)", "PredictionTower/conv2d_3/activation_0/Relu6:0"),
    ("FPN P4 (20x20x128)", "PredictionTower/conv2d_3/activation_1/Relu6:0"),
    ("FPN P5 (10x10x128)", "PredictionTower/conv2d_3/activation_2/Relu6:0"),
    ("FPN P6 (5x5x128)", "PredictionTower/conv2d_3/activation_3/Relu6:0"),
    ("FPN P7 (3x3x128)", "PredictionTower/conv2d_3/activation_4/Relu6:0"),
]

resolved_layers = []
for label, suffix in LAYER_PATTERNS:
    match = next((name for name in all_tensor_names if name.endswith(suffix)), None)
    if match:
        resolved_layers.append((label, match))

if not resolved_layers:
    raise RuntimeError("Tensor feature map MobileNetV2/FPNLite tidak ditemukan pada SavedModel ini.")

fetch_tensors = [frozen_fn.graph.get_tensor_by_name(name) for _, name in resolved_layers]
feature_extractor_fn = frozen_fn.prune(frozen_fn.inputs, fetch_tensors)

input_batch = tf.convert_to_tensor(proc_rgb[None, ...], dtype=tf.uint8)
raw_fmaps = feature_extractor_fn(input_batch)
feature_maps = {label: t.numpy()[0] for (label, _), t in zip(resolved_layers, raw_fmaps)}

# Tabel ringkasan statistik fitur tiap layer
stats_rows = []
for label, fmap in feature_maps.items():
    h_f, w_f, c_f = fmap.shape
    stats_rows.append({
        "Tahap_Layer": label,
        "Resolusi_Spasial": f"{h_f}x{w_f}",
        "Jumlah_Channel": c_f,
        "Total_Nilai_Fitur": h_f * w_f * c_f,
        "Aktivasi_Mean": round(float(fmap.mean()), 4),
        "Aktivasi_Max": round(float(fmap.max()), 4),
    })
df_fmap_stats = pd.DataFrame(stats_rows)
display(df_fmap_stats)

# Plot 1: Visualisasi Feature Maps (Mean aktivasi & 3 channel terkuat per tahap utama)
selected_for_grid = [k for k in feature_maps.keys() if not k.startswith("FPN P7")]
fig, axes = plt.subplots(len(selected_for_grid), 5, figsize=(16, 2.7 * len(selected_for_grid)))

for r, label in enumerate(selected_for_grid):
    fmap = feature_maps[label]  # shape: (H, W, C)
    mean_map = fmap.mean(axis=-1)
    ch_energy = fmap.mean(axis=(0, 1))
    top_channels = np.argsort(-ch_energy)[:4]

    axes[r, 0].imshow(mean_map, cmap="viridis")
    axes[r, 0].set_title(f"{label}\nMean Semua Channel", fontsize=9)
    for c_idx, ch in enumerate(top_channels, start=1):
        axes[r, c_idx].imshow(fmap[:, :, ch], cmap="magma")
        axes[r, c_idx].set_title(f"Channel #{ch}\n(mean={ch_energy[ch]:.2f})", fontsize=8.5)
    for c in range(5):
        axes[r, c].axis("off")

fig.suptitle(f"Tahap 2: Ekstraksi Fitur Otomatis MobileNetV2 + FPNLite — {active_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "02_feature_maps_mobilenet.png")
plt.show()


# Plot 2: Activation Heatmap Overlay (menggabungkan skala P3, P4, P5 ke resolusi asli)
def build_fpn_heatmap(fmaps_dict: dict, target_hw: tuple) -> np.ndarray:
    H, W = target_hw
    acc = np.zeros((H, W), dtype=np.float32)
    used = 0
    for key in ("FPN P3 (40x40x128)", "FPN P4 (20x20x128)", "FPN P5 (10x10x128)"):
        if key in fmaps_dict:
            m = fmaps_dict[key].mean(axis=-1)
            m = (m - m.min()) / (m.max() - m.min() + 1e-6)
            acc += cv2.resize(m, (W, H), interpolation=cv2.INTER_CUBIC)
            used += 1
    acc /= max(1, used)
    acc = (acc - acc.min()) / (acc.max() - acc.min() + 1e-6)
    return acc


heatmap_norm = build_fpn_heatmap(feature_maps, proc_rgb.shape[:2])
heatmap_color = cv2.cvtColor(cv2.applyColorMap((heatmap_norm * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
heatmap_overlay = cv2.addWeighted(proc_rgb, 0.55, heatmap_color, 0.45, 0)

fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4.8))
axes2[0].imshow(proc_rgb)
axes2[0].set_title("1. Citra Masukan")
axes2[1].imshow(heatmap_norm, cmap="jet")
axes2[1].set_title("2. Peta Aktivasi FPNLite (P3+P4+P5)")
axes2[2].imshow(heatmap_overlay)
axes2[2].set_title("3. Overlay Fokus Perhatian MobileNetV2")
for ax in axes2:
    ax.axis("off")
fig2.suptitle(f"Visualisasi Fokus Ekstraksi Fitur CNN — {active_name}", fontsize=13.5, fontweight="bold")
plt.tight_layout()
save_fig(fig2, "03_activation_heatmap.png")
plt.show()

## Cell 5 — Tahap 3: Deteksi & Post-Processing (Raw Anchors → Confidence Threshold → NMS)

Mengikuti `apps/computer-vision/src/ferqueue_cv/postprocessing.py` dan `receiver-v3.json`:
1. **Raw Predictions**: SSD menghasilkan **12.804 kandidat anchor** dari piramida fitur P3–P7 (kita tampilkan 40 kandidat teratas sebelum disaring).
2. **Confidence Threshold (`>= 0.50`)**: membuang kandidat latar belakang dengan skor rendah.
3. **Class-Agnostic NMS (`IoU <= 0.45`)**: menghapus kotak duplikat yang saling tumpang tindih pada objek yang sama hingga menyisakan **Bounding Box akhir + Label Kelas + Confidence Score**.

In [ ]:
CONFIDENCE_THRESHOLD = 0.50
NMS_IOU_THRESHOLD = 0.45
MAX_DETECTIONS = 10
CLASS_AGNOSTIC_NMS = True

CLASS_COLORS = {
    "Mobil": (40, 210, 90),
    "Truk": (255, 165, 0),
    "Bus": (65, 155, 255),
}


def select_detections_ferqueue(result: dict, conf_th=0.5, iou_th=0.45, max_det=10, class_agnostic=True):
    """Persis mengikuti apps/computer-vision/src/ferqueue_cv/postprocessing.py."""
    boxes = np.asarray(result["detection_boxes"])
    scores = np.asarray(result["detection_scores"])
    classes = np.asarray(result["detection_classes"])
    candidates = [
        int(idx)
        for idx in np.argsort(-scores, kind="stable")
        if scores[idx] >= conf_th
        and np.isfinite(scores[idx])
        and np.isfinite(boxes[idx]).all()
        and np.isfinite(classes[idx])
        and boxes[idx][2] > boxes[idx][0]
        and boxes[idx][3] > boxes[idx][1]
    ]
    keep = []
    while candidates and len(keep) < max_det:
        current = candidates.pop(0)
        keep.append(current)
        remaining = []
        a = boxes[current]
        for idx in candidates:
            if not class_agnostic and classes[idx] != classes[current]:
                remaining.append(idx)
                continue
            b = boxes[idx]
            inter = max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
            union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
            if union <= 0 or inter / union <= iou_th:
                remaining.append(idx)
        candidates = remaining
    return keep


def run_ssd_inference(rgb: np.ndarray):
    t_start = perf_counter()
    raw_out = detector_fn(tf.convert_to_tensor(rgb[None, ...], dtype=tf.uint8))
    elapsed_ms = (perf_counter() - t_start) * 1000.0
    count = int(raw_out["num_detections"][0])
    parsed = {
        "detection_boxes": raw_out["detection_boxes"][0, :count].numpy(),
        "detection_scores": raw_out["detection_scores"][0, :count].numpy(),
        "detection_classes": raw_out["detection_classes"][0, :count].numpy(),
        "raw_detection_boxes": raw_out["raw_detection_boxes"][0].numpy(),
        "raw_detection_scores": raw_out["raw_detection_scores"][0].numpy(),
        "inference_ms": elapsed_ms,
    }
    return parsed


# Warmup 1x agar catatan waktu inferensi akurat
_ = detector_fn(tf.convert_to_tensor(proc_rgb[None, ...], dtype=tf.uint8))
ssd_out = run_ssd_inference(proc_rgb)

H, W = proc_rgb.shape[:2]

# Panel 1: Top-40 raw anchor candidates (sebelum NMS internal/eksternal)
raw_scores_fg = ssd_out["raw_detection_scores"][:, 1:].max(axis=-1)
top40_idx = np.argsort(-raw_scores_fg)[:40]
vis_raw = proc_rgb.copy()
for i in top40_idx:
    y1, x1, y2, x2 = ssd_out["raw_detection_boxes"][i]
    cv2.rectangle(vis_raw, (int(x1 * W), int(y1 * H)), (int(x2 * W), int(y2 * H)), (255, 80, 80), 1)

# Panel 2: Lolos Confidence Threshold (>= 0.5) sebelum NMS proyek
vis_pre_nms = proc_rgb.copy()
pre_nms_idx = [i for i, s in enumerate(ssd_out["detection_scores"]) if s >= CONFIDENCE_THRESHOLD]
for i in pre_nms_idx:
    y1, x1, y2, x2 = ssd_out["detection_boxes"][i]
    cv2.rectangle(vis_pre_nms, (int(x1 * W), int(y1 * H)), (int(x2 * W), int(y2 * H)), (255, 215, 0), 2)

# Panel 3: Hasil Akhir setelah Confidence + Class-Agnostic NMS
kept_idx = select_detections_ferqueue(
    ssd_out, CONFIDENCE_THRESHOLD, NMS_IOU_THRESHOLD, MAX_DETECTIONS, CLASS_AGNOSTIC_NMS
)
vis_final = proc_rgb.copy()
final_detections = []
for i in kept_idx:
    y1, x1, y2, x2 = map(float, ssd_out["detection_boxes"][i])
    score = float(ssd_out["detection_scores"][i])
    cls_id = int(ssd_out["detection_classes"][i])
    cls_name = CLASSES[cls_id - 1] if 1 <= cls_id <= len(CLASSES) else f"ID-{cls_id}"
    color = CLASS_COLORS.get(cls_name, (40, 210, 90))
    px1, py1 = max(0, int(x1 * W)), max(0, int(y1 * H))
    px2, py2 = min(W, int(x2 * W)), min(H, int(y2 * H))
    cv2.rectangle(vis_final, (px1, py1), (px2, py2), color, 3)
    label_txt = f"{cls_name} {score:.2f}"
    (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(vis_final, (px1, max(0, py1 - th - 10)), (px1 + tw + 8, max(th + 10, py1)), color, -1)
    cv2.putText(vis_final, label_txt, (px1 + 4, max(th + 4, py1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2, cv2.LINE_AA)
    final_detections.append({
        "Kelas": cls_name,
        "Confidence": round(score, 4),
        "box_norm": {"ymin": y1, "xmin": x1, "ymax": y2, "xmax": x2},
        "BBox_px": f"({px1}, {py1}, {px2 - px1}x{py2 - py1})",
        "Inferensi_ms": round(ssd_out["inference_ms"], 1),
    })

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.2))
axes[0].imshow(vis_raw)
axes[0].set_title(f"1. Raw Anchor Candidates\n(Top 40 dari {len(raw_scores_fg):,} anchor)")
axes[1].imshow(vis_pre_nms)
axes[1].set_title(f"2. Setelah Confidence >= {CONFIDENCE_THRESHOLD}\n({len(pre_nms_idx)} kandidat sebelum NMS)")
axes[2].imshow(vis_final)
axes[2].set_title(f"3. Hasil Akhir Setelah NMS (IoU<={NMS_IOU_THRESHOLD})\n({len(final_detections)} deteksi | {ssd_out['inference_ms']:.1f} ms)")
for ax in axes:
    ax.axis("off")

fig.suptitle(f"Tahap 3: Deteksi & Post-Processing SSD MobileNetV2 — {active_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "04_raw_vs_nms_deteksi.png")
plt.show()

if final_detections:
    df_det = pd.DataFrame([{k: v for k, v in d.items() if k != "box_norm"} for d in final_detections])
    display(df_det)
else:
    print("Tidak ada objek yang melewati CONFIDENCE_THRESHOLD. Coba turunkan nilainya (misal 0.30).")

## Cell 6 — Tahap 4: Segmentasi Pasca-Deteksi (GrabCut + Morfologi sesuai `processing.py`)

Sesuai dokumentasi proyek FERQUEUE (`docs/PCV_MANUAL.md` & `ferqueue_cv/processing.py`):
* Model SSD MobileNetV2 **tidak memerlukan segmentasi piksel sebelum deteksi** (gambar utuh langsung masuk ke jaringan).
* Namun setelah **Bounding Box** diperoleh dari SSD, proyek ini menyediakan **GrabCut + Operasi Morfologi (Opening & Closing)** untuk memisahkan siluet *foreground* kendaraan dari *background* (sebagai tahap review / analisis bentuk objek).

In [ ]:
GRABCUT_ITERATIONS = 5
GRABCUT_MORPH_KERNEL = 3


def grabcut_mask_ferqueue(rgb: np.ndarray, box_norm: dict, iterations=5, morph_k=3):
    """Persis mengikuti grabcut_mask() di apps/computer-vision/src/ferqueue_cv/processing.py."""
    height, width = rgb.shape[:2]
    x1 = max(0, math.floor(box_norm["xmin"] * width))
    y1 = max(0, math.floor(box_norm["ymin"] * height))
    x2 = min(width, math.ceil(box_norm["xmax"] * width))
    y2 = min(height, math.ceil(box_norm["ymax"] * height))
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None, "box_too_small"
    padded = cv2.copyMakeBorder(rgb, 1, 1, 1, 1, cv2.BORDER_REPLICATE)
    mask = np.zeros(padded.shape[:2], dtype=np.uint8)
    cv2.setRNGSeed(42)
    try:
        cv2.grabCut(
            padded, mask, (x1 + 1, y1 + 1, x2 - x1, y2 - y1),
            np.zeros((1, 65)), np.zeros((1, 65)), iterations, cv2.GC_INIT_WITH_RECT,
        )
    except cv2.error:
        return None, "grabcut_failed"
    fg = np.isin(mask[1:-1, 1:-1], (cv2.GC_FGD, cv2.GC_PR_FGD)).astype(np.uint8) * 255
    if morph_k:
        elem = np.ones((morph_k, morph_k), dtype=np.uint8)
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, elem)
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, elem)
    bounded = np.zeros_like(fg)
    bounded[y1:y2, x1:x2] = fg[y1:y2, x1:x2]
    if not bounded.any():
        return None, "empty_mask"
    return bounded, "ok"


combined_grabcut = np.zeros(proc_rgb.shape[:2], dtype=np.uint8)
for det in final_detections:
    m, status = grabcut_mask_ferqueue(proc_rgb, det["box_norm"], GRABCUT_ITERATIONS, GRABCUT_MORPH_KERNEL)
    if m is not None:
        combined_grabcut = cv2.bitwise_or(combined_grabcut, m)

cutout_gc = cv2.bitwise_and(proc_rgb, proc_rgb, mask=combined_grabcut)
overlay_gc = proc_rgb.copy().astype(np.float32)
tint = np.zeros_like(overlay_gc)
tint[:, :] = (255, 70, 20)
m_bool = combined_grabcut > 0
overlay_gc[m_bool] = 0.55 * overlay_gc[m_bool] + 0.45 * tint[m_bool]
overlay_gc = overlay_gc.astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
axes[0].imshow(vis_final)
axes[0].set_title("1. ROI dari Bbox SSD")
axes[1].imshow(combined_grabcut, cmap="gray")
axes[1].set_title("2. Mask GrabCut + Morfologi")
axes[2].imshow(cutout_gc)
axes[2].set_title("3. Cutout Foreground")
axes[3].imshow(overlay_gc)
axes[3].set_title("4. Overlay Segmentasi")
for ax in axes:
    ax.axis("off")

fig.suptitle(f"Tahap 4: Segmentasi Pasca-Deteksi (GrabCut Dipandu Bbox SSD) — {active_name}", fontsize=13.5, fontweight="bold")
plt.tight_layout()
save_fig(fig, "05_segmentasi_grabcut.png")
plt.show()

## Cell 7 — Ringkasan Alur 1-Baris Proyek FERQUEUE (Siap PPT!) & Multi-Gambar

Membuat 1 gambar ringkasan horizontal 5-panel dari awal sampai akhir yang langsung siap ditempel di PowerPoint, lalu menjalankan deteksi pada semua gambar uji yang kamu muat.

In [ ]:
# 1. Ringkasan Alur 1-Baris (5 Panel)
fig, axes = plt.subplots(1, 5, figsize=(20, 4.3))
summary_panels = [
    ("1. Input Citra\n(RGB Asli)", proc_rgb),
    ("2. Preprocessing\n(Resize 320x320, [-1,1])", resized_320),
    ("3. Ekstraksi Fitur\n(MobileNetV2+FPNLite)", heatmap_overlay),
    ("4. Hasil Deteksi\n(Bbox + Label + NMS)", vis_final),
    ("5. Segmentasi ROI\n(GrabCut + Morfologi)", overlay_gc),
]
for ax, (title, im) in zip(axes, summary_panels):
    ax.imshow(im)
    ax.set_title(title, fontsize=10.5)
    ax.axis("off")

fig.suptitle(f"Ringkasan Alur Proyek FERQUEUE (SSD MobileNetV2 FPNLite) — {active_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "06_alur_lengkap_ferqueue.png")
plt.show()

# 2. Evaluasi pada semua gambar uji yang dimuat
if len(loaded_images) > 1:
    cols = min(len(loaded_images), 3)
    rows = math.ceil(len(loaded_images) / cols)
    fig_m, axes_m = plt.subplots(rows, cols, figsize=(5.2 * cols, 4.2 * rows))
    axes_m = np.atleast_1d(axes_m).ravel()
    for idx, ax in enumerate(axes_m):
        if idx < len(loaded_images):
            nm, pil_i = loaded_images[idx]
            arr_i = np.asarray(preprocess_image_ferqueue(pil_i, MODEL_PREPROCESSING), dtype=np.uint8)
            out_i = run_ssd_inference(arr_i)
            keep_i = select_detections_ferqueue(out_i, CONFIDENCE_THRESHOLD, NMS_IOU_THRESHOLD, MAX_DETECTIONS, CLASS_AGNOSTIC_NMS)
            vis_i = arr_i.copy()
            hi, wi = arr_i.shape[:2]
            labels_found = []
            for k in keep_i:
                y1, x1, y2, x2 = map(float, out_i["detection_boxes"][k])
                sc = float(out_i["detection_scores"][k])
                cid = int(out_i["detection_classes"][k])
                cname = CLASSES[cid - 1] if 1 <= cid <= len(CLASSES) else str(cid)
                col_c = CLASS_COLORS.get(cname, (40, 210, 90))
                cv2.rectangle(vis_i, (int(x1 * wi), int(y1 * hi)), (int(x2 * wi), int(y2 * hi)), col_c, 3)
                cv2.putText(vis_i, f"{cname} {sc:.2f}", (int(x1 * wi) + 4, max(24, int(y1 * hi) + 22)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, col_c, 2, cv2.LINE_AA)
                labels_found.append(f"{cname}({sc:.2f})")
            ax.imshow(vis_i)
            ax.set_title(f"{nm[:22]}\n{', '.join(labels_found) or 'Tidak terdeteksi'} ({out_i['inference_ms']:.0f}ms)", fontsize=9)
        ax.axis("off")
    fig_m.suptitle("Hasil Deteksi SSD MobileNetV2 pada Semua Gambar Uji", fontsize=13.5, fontweight="bold")
    plt.tight_layout()
    save_fig(fig_m, "07_multi_gambar_ferqueue.png")
    plt.show()

## Cell 8 — *(Opsional)* Modul Kalibrasi Koin 50 Yen (`side_measurement.py`) & Unduh ZIP Output PPT

Di proyek FERQUEUE juga terdapat modul `apps/computer-vision/src/ferqueue_cv/side_measurement.py` yang memakai **PCV Klasik (Grayscale → Gaussian Blur → Otsu Thresholding → Contour → Ellipse Fitting)** pada koin referensi 50 yen (`diameter = 21.0 mm`) untuk mengestimasi panjang dan tinggi fisik kendaraan (dalam `cm`).

* Ubah `RUN_COIN_DEMO = True` dan atur `COIN_REGION` (`ymin, xmin, ymax, xmax` ternormalisasi `0..1`) jika foto ujimu memiliki koin 50 yen.
* Di bagian bawah, semua gambar PNG (300 DPI) dikemas menjadi `nb2_ferqueue_mobilenet_outputs.zip`.

In [ ]:
RUN_COIN_DEMO = False
COIN_REGION = (0.70, 0.05, 0.98, 0.30)  # (ymin, xmin, ymax, xmax) area koin pada gambar
COIN_DIAMETER_MM = 21.0

if RUN_COIN_DEMO:
    y1c, x1c, y2c, x2c = COIN_REGION
    left, top = int(x1c * W), int(y1c * H)
    right, bottom = math.ceil(x2c * W), math.ceil(y2c * H)
    crop_gray = cv2.cvtColor(proc_rgb[top:bottom, left:right], cv2.COLOR_RGB2GRAY)
    crop_h, crop_w = crop_gray.shape[:2]
    blur_c = cv2.GaussianBlur(crop_gray, (5, 5), 0)
    _, otsu_c = cv2.threshold(blur_c, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    candidates = []
    if min(crop_w, crop_h) >= 28:
        for binary in (otsu_c, cv2.bitwise_not(otsu_c)):
            cnts_c, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in cnts_c:
                _, _, wc, hc = cv2.boundingRect(cnt)
                if len(cnt) < 5 or min(wc, hc) < 24:
                    continue
                (cx, cy), axes_e, _ = cv2.fitEllipse(cnt)
                minor, major = sorted(axes_e)
                if not (24 <= minor <= major and minor / major >= 0.88):
                    continue
                margin = major / 2.0 + 2.0
                if not (margin <= cx <= crop_w - margin and margin <= cy <= crop_h - margin):
                    continue
                if major < 0.4 * min(crop_w, crop_h):
                    continue
                fill = cv2.contourArea(cnt) / (math.pi * minor * major / 4.0)
                if not 0.82 <= fill <= 1.12:
                    continue
                diam = major + 1.0
                candidates.append({"cx": cx, "cy": cy, "diameter_px": diam, "mm_per_px": COIN_DIAMETER_MM / diam})
    coin_found = candidates[0] if len(candidates) == 1 else None

    fig_c, axes_c = plt.subplots(1, 3, figsize=(12, 3.8))
    axes_c[0].imshow(crop_gray, cmap="gray")
    axes_c[0].set_title("1. Crop ROI Koin (Grayscale)")
    axes_c[1].imshow(otsu_c, cmap="gray")
    axes_c[1].set_title("2. Segmentasi Otsu Threshold")
    vis_coin = cv2.cvtColor(crop_gray, cv2.COLOR_GRAY2RGB)
    if coin_found:
        cv2.circle(vis_coin, (int(coin_found["cx"]), int(coin_found["cy"])), int(coin_found["diameter_px"] / 2), (40, 220, 80), 2)
        axes_c[2].set_title(f"3. Koin Terdeteksi\nd={coin_found['diameter_px']:.1f}px ({coin_found['mm_per_px']:.3f} mm/px)")
        if len(final_detections) == 1:
            b = final_detections[0]["box_norm"]
            cm_per_px = coin_found["mm_per_px"] / 10.0
            L_cm = (b["xmax"] - b["xmin"]) * W * cm_per_px
            H_cm = (b["ymax"] - b["ymin"]) * H * cm_per_px
            print(f"Estimasi Dimensi Samping Kendaraan: Panjang={L_cm:.2f} cm | Tinggi={H_cm:.2f} cm")
    else:
        axes_c[2].set_title("3. Koin Tidak Terdeteksi di ROI")
    axes_c[2].imshow(vis_coin)
    for ax in axes_c:
        ax.axis("off")
    plt.tight_layout()
    save_fig(fig_c, "08_kalibrasi_koin_otsu.png")
    plt.show()

zip_path = OUT_DIR.parent / "nb2_ferqueue_mobilenet_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"Semua gambar PPT tersimpan di: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")
# Hapus tanda '#' di bawah jika ingin langsung mengunduh ZIP di Colab:
# if IN_COLAB:
#     colab_files.download(str(zip_path))